In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
import warnings
warnings.filterwarnings('ignore')


path = '/content/drive/MyDrive/Thesis file/Feature_Extract_Formants/'

extractors = ['AlexNet', 'CustomCNN', 'DenseNet121', 'EfficientNetB0', 'InceptionV3',
              'MobileNetV2', 'ResNet50', 'VGG16', 'ViT', 'Xception']

classifiers = {
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(random_state=42)
}

results = pd.DataFrame(columns=['Feature Extractor'] + list(classifiers.keys()) + ['Best', 'Best Accuracy'])
results['Feature Extractor'] = extractors

for extractor in extractors:
    print(f'\nProcessing {extractor} ...')

    # F1 থেকে লেবেল নিব (যেটা কাজ করেছে)
    f1_file = f'{path}F1_Features_{extractor}.csv'
    f2_file = f'{path}F2_Features_{extractor}.csv'
    f4_file = f'{path}F4_Features_{extractor}.csv'   # ← এখানে F4

    df1 = pd.read_csv(f1_file)  # লেবেলের জন্য
    df2 = pd.read_csv(f2_file)
    df4 = pd.read_csv(f4_file)  # ← F4

    # লেবেল কলাম বের করি F1 থেকে
    possible_label_cols = ['label', 'class', 'target', 'phoneme', 'category', 'Label', 'Class', 'phoneme_label']
    label_col = None
    for col in possible_label_cols:
        if col in df1.columns:
            label_col = col
            break

    if label_col is None:
        print(f"   → F1 এ লেবেল পাওয়া যায়নি! স্কিপ।")
        continue

    print(f"   → লেবেল পাওয়া গেছে: {label_col} → ক্লাস: {df1[label_col].nunique()}টা")

    y = df1[label_col]

    # ক্লিন ফাংশন
    def clean(df):
        df = df.copy()
        drop_cols = [c for c in df.columns if any(x in c.lower() for x in ['filename', 'path', 'unnamed', 'index'])]
        df.drop(columns=drop_cols, errors='ignore', inplace=True)
        return df

    df2_clean = clean(df2)
    df4_clean = clean(df4)

    # লেবেল কলাম থাকলে ড্রপ করি
    if label_col in df2_clean.columns:
        df2_clean = df2_clean.drop(columns=[label_col])
    if label_col in df4_clean.columns:
        df4_clean = df4_clean.drop(columns=[label_col])

    # F2 + F4 কনক্যাট
    X = pd.concat([df2_clean.reset_index(drop=True), df4_clean.reset_index(drop=True)], axis=1)

    # স্প্লিট
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
        print(f"   → Stratified স্প্লিট হয়েছে")
    except:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        print(f"   → সাধারণ স্প্লিট")

    best_acc = 0
    best_name = ''

    for name, clf in classifiers.items():
        try:
            clf.fit(X_train, y_train)
            acc = round(accuracy_score(y_test, clf.predict(X_test)), 4)
            results.loc[results['Feature Extractor'] == extractor, name] = acc
            if acc > best_acc:
                best_acc = acc
                best_name = name
            print(f"   → {name}: {acc}")
        except Exception as e:
            print(f"   → {name} ফেইল: {e}")
            results.loc[results['Feature Extractor'] == extractor, name] = 0.0

    results.loc[results['Feature Extractor'] == extractor, 'Best'] = best_name
    results.loc[results['Feature Extractor'] == extractor, 'Best Accuracy'] = best_acc

# সেভ করি
output_file = path + 'Full_Summary_Table_F2_F4.xlsx'
results.to_excel(output_file, index=False)

print('\n' + '='*60)
print('সব শেষ! F2 + F4 কম্বিনেশনের রেজাল্ট তৈরি হয়ে গেছে!')
print(f"ফাইল সেভ হয়েছে: {output_file}")
display(results)

Mounted at /content/drive

Processing AlexNet ...
   → লেবেল পাওয়া গেছে: label → ক্লাস: 2টা
   → Stratified স্প্লিট হয়েছে
   → AdaBoost: 0.8857
   → Decision Tree: 0.7714
   → Extra Trees: 0.8571
   → Gradient Boosting: 0.8952
   → KNN: 0.8667
   → Logistic Regression: 0.9333
   → Naive Bayes: 0.8286
   → Random Forest: 0.8857
   → SVM: 0.9333

Processing CustomCNN ...
   → লেবেল পাওয়া গেছে: label → ক্লাস: 2টা
   → Stratified স্প্লিট হয়েছে
   → AdaBoost: 0.8952
   → Decision Tree: 0.7429
   → Extra Trees: 0.8381
   → Gradient Boosting: 0.8762
   → KNN: 0.781
   → Logistic Regression: 0.9143
   → Naive Bayes: 0.6571
   → Random Forest: 0.8571
   → SVM: 0.8762

Processing DenseNet121 ...
   → লেবেল পাওয়া গেছে: label → ক্লাস: 2টা
   → Stratified স্প্লিট হয়েছে
   → AdaBoost: 0.8857
   → Decision Tree: 0.8476
   → Extra Trees: 0.8857
   → Gradient Boosting: 0.9048
   → KNN: 0.8762
   → Logistic Regression: 0.9143
   → Naive Bayes: 0.9048
   → Random Forest: 0.8857
   → SVM: 0.9048

Pr

,Feature Extractor,AdaBoost,Decision Tree,Extra Trees,Gradient Boosting,KNN,Logistic Regression,Naive Bayes,Random Forest,SVM,Best,Best Accuracy
0,AlexNet,0.8857,0.7714,0.8571,0.8952,0.8667,0.9333,0.8286,0.8857,0.9333,Logistic Regression,0.9333
1,CustomCNN,0.8952,0.7429,0.8381,0.8762,0.781,0.9143,0.6571,0.8571,0.8762,Logistic Regression,0.9143
2,DenseNet121,0.8857,0.8476,0.8857,0.9048,0.8762,0.9143,0.9048,0.8857,0.9048,Logistic Regression,0.9143
3,EfficientNetB0,0.9143,0.8286,0.8857,0.9048,0.8857,0.9143,0.8762,0.8571,0.8952,AdaBoost,0.9143
4,InceptionV3,0.8762,0.7619,0.8762,0.8667,0.819,0.9048,0.8,0.8381,0.8667,Logistic Regression,0.9048
5,MobileNetV2,0.8667,0.8286,0.8571,0.8952,0.8952,0.9238,0.8857,0.8667,0.9524,SVM,0.9524
6,ResNet50,0.8952,0.8,0.8952,0.9238,0.8476,0.9333,0.8667,0.9048,0.9048,Logistic Regression,0.9333
7,VGG16,0.9143,0.781,0.8667,0.9048,0.9143,0.9238,0.7619,0.8762,0.9429,SVM,0.9429
8,ViT,0.8286,0.7429,0.819,0.8381,0.8095,0.8667,0.8286,0.8286,0.819,Logistic Regression,0.8667
9,Xception,0.8952,0.819,0.8667,0.8667,0.819,0.9143,0.8667,0.8667,0.8476,Logistic Regression,0.9143
